# Práctica 3: Modelos Predictivos y Minería de Datos
**Objetivo del modelo:** El objetivo principal de este modelo predictivo es clasificar el riesgo de que una paciente padezca diabetes (Outcome = 1) o no (Outcome = 0), basándose en 6 indicadores clínicos clave.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import cross_validate, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# 1. Cargamos directamente los datos ya normalizados
df = pd.read_excel('datos_normalizados.xlsx')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

variables_entrenamiento = X.columns.tolist()
print("Variables predictoras:", variables_entrenamiento)

### (0.5) Selección de Factores

In [ ]:
plt.figure(figsize=(8, 6))
correlaciones = df.corr()
sns.heatmap(correlaciones, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de Correlación Final")
plt.show()

print("Impacto de cada factor en la Diabetes (Outcome):")
print(correlaciones['Outcome'].sort_values(ascending=False)[1:])

### (1.5) Configuración y Evaluación de Modelos (Revisión Overfitting/Underfitting)
Para asegurar que los modelos sean capaces de generalizar y no memoricen los datos (F1 Train = 1.0), se han añadido parámetros de regularización estrictos desde el principio (como `max_depth` y `min_samples_leaf` en los árboles). El objetivo es que la diferencia entre Train y Test no supere los 0.05 puntos.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = ('f1_macro', 'accuracy', 'precision_macro', 'recall_macro')

# Modelos configurados para evitar la memorización de datos (Overfitting)
modelos = {
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=3, min_samples_leaf=15, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Red Neuronal": MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42),
    "SVM": SVC(kernel='rbf', C=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=10, random_state=42),
    "XGBoost": XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=100, random_state=42, eval_metric='logloss')
}

df_comparacion = pd.DataFrame(columns=['Modelo', 'F1_Train', 'F1_Test', 'Diferencia (Overfitting)'])

for nombre, modelo in modelos.items():
    scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring, return_train_score=True)
    f1_train = scores['train_f1_macro'].mean()
    f1_test = scores['test_f1_macro'].mean()
    dif = f1_train - f1_test
    df_comparacion.loc[len(df_comparacion)] = [nombre, f1_train, f1_test, dif]

print("Evaluación de Calidad de los Modelos (Sin Sobreajuste):")
display(df_comparacion.sort_values(by='F1_Test', ascending=False))

### (1.0) Hiperparametrización con GridSearch
Realizamos una búsqueda exhaustiva (GridSearch) en Random Forest, restringiendo deliberadamente su complejidad para garantizar una excelente generalización ante nuevos datos.

In [ ]:
rf_parametros = {
    'n_estimators': [50, 100],
    'max_depth': [2, 3, 4], # Límite estricto de profundidad
    'min_samples_leaf': [10, 15, 20] # Requiere un mínimo alto de muestras por hoja
}

rf_modelo = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf_modelo, param_grid=rf_parametros, scoring='f1_macro', cv=cv, n_jobs=-1)
grid_search.fit(X, y)

mejor_modelo = grid_search.best_estimator_
print("Mejores parámetros encontrados:", grid_search.best_params_)
print("Mejor F1-score CV (Test):", grid_search.best_score_)

# Generamos el scaler a partir del CSV original para usarlo en la app
from sklearn.preprocessing import MinMaxScaler
raw_data = pd.read_csv("pima-indians-diabetes.data.csv", header=None, 
                       names=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'])
X_raw = raw_data[variables_entrenamiento]
scaler = MinMaxScaler()
scaler.fit(X_raw)

# EXPORTAR MODELO Y SCALER LISTOS PARA PRODUCCIÓN
with open('modelo_diabetes_optimizado.pkl', 'wb') as file:
    pickle.dump((mejor_modelo, variables_entrenamiento, scaler), file)
print("\n¡Modelo y Scaler exportados correctamente para Streamlit!")